In [ ]:


import os
import sys
import numpy as np
import matplotlib.pyplot as plt

# -----------------------

IN_COLAB = False
try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    IN_COLAB = False

# -----------------------

SEED = 42
rng = np.random.default_rng(SEED)

# -----------------------

year_start = 2024
year_end   = 2050
years      = np.arange(year_start, year_end + 1)
Y          = len(years)

days_per_year = 365


D2_2024 = 1.0
D1_2024 = 0.6


c_F   = 70.0       # benchmark off-peak price level (AUD/MWh)
p_cap = 500.0      # peak price cap (AUD/MWh) for the model's smooth rule
eta   = 0.9        # round-trip efficiency

Delta = p_cap - c_F


K_R0 = 0.64 * D2_2024   # total effective solar capacity ~ 64% of peak
K_S0 = 0.03 * D2_2024   # storage energy capacity ~ 3% of peak

# -----------------------

USE_OBSERVED = True
obs_peak_2024 = None
obs_peak_2025 = None

def compute_observed_peak_prices_local_4_8pm(start_time, end_time, raw_cache_dir):
    """
    Returns (obs_2024, obs_2025) where each is the NEM-wide demand-weighted average
    dispatch RRP over the LOCAL-time window [16:00, 20:00), computed separately for 2024 and 2025.

    Demand-weighted definition (across all regions & intervals in-window):
        avg = sum(rrp * demand) / sum(demand)

    Notes:
    - AEMO dispatch timestamps are provided in NEM time (AEST). We localize as Australia/Brisbane,
      then convert to each region's local timezone (DST-aware) to filter 16:00–20:00 local.
    """
    import importlib.util, subprocess, inspect
    if importlib.util.find_spec("nemosis") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "nemosis"])

    from nemosis import dynamic_data_compiler
    import pandas as pd

    os.makedirs(raw_cache_dir, exist_ok=True)


    def ddc(table_name):
        params = inspect.signature(dynamic_data_compiler).parameters
        if "raw_data_cache" in params:
            return dynamic_data_compiler(
                start_time, end_time,
                table_name=table_name,
                raw_data_cache=raw_cache_dir
            )
        if "raw_data_location" in params:
            return dynamic_data_compiler(
                start_time, end_time,
                table_name=table_name,
                raw_data_location=raw_cache_dir
            )

        return dynamic_data_compiler(start_time, end_time, table_name, raw_cache_dir)

    price = ddc("DISPATCHPRICE")
    demand = ddc("DISPATCHREGIONSUM")

    def pick_col(df, candidates):
        for c in candidates:
            if c in df.columns:
                return c
        raise KeyError(f"None of these columns found: {candidates}")

    tcol_p = pick_col(price,  ["SETTLEMENTDATE", "INTERVAL_DATETIME", "DATETIME"])
    rcol_p = pick_col(price,  ["REGIONID", "REGION"])
    pcol   = pick_col(price,  ["RRP", "PRICE", "DISPATCHPRICE"])

    tcol_d = pick_col(demand, ["SETTLEMENTDATE", "INTERVAL_DATETIME", "DATETIME"])
    rcol_d = pick_col(demand, ["REGIONID", "REGION"])
    dcol   = pick_col(demand, ["TOTALDEMAND", "DEMAND", "OPERATIONALDEMAND"])

    price = price[[tcol_p, rcol_p, pcol]].copy()
    demand = demand[[tcol_d, rcol_d, dcol]].copy()

    price.rename(columns={tcol_p: "time", rcol_p: "region", pcol: "rrp"}, inplace=True)
    demand.rename(columns={tcol_d: "time", rcol_d: "region", dcol: "demand"}, inplace=True)

    price["time"] = pd.to_datetime(price["time"])
    demand["time"] = pd.to_datetime(demand["time"])


    price["time"]  = price["time"].dt.tz_localize("Australia/Brisbane")
    demand["time"] = demand["time"].dt.tz_localize("Australia/Brisbane")

    df = price.merge(demand, on=["time", "region"], how="inner")
    df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=["rrp", "demand"])
    df = df[df["demand"] > 0]

    REGION_TZ = {
        "NSW1": "Australia/Sydney",
        "VIC1": "Australia/Melbourne",
        "QLD1": "Australia/Brisbane",
        "SA1":  "Australia/Adelaide",
        "TAS1": "Australia/Hobart",
    }

    out = []
    for reg, tz in REGION_TZ.items():
        g = df[df["region"] == reg].copy()
        if len(g) == 0:
            continue
        g["local_time"] = g["time"].dt.tz_convert(tz)
        h = g["local_time"].dt.hour
        g = g[(h >= 16) & (h < 20)]
        out.append(g)

    if len(out) == 0:
        raise RuntimeError("No data remained after local-time peak window filtering.")

    df_peak = pd.concat(out, ignore_index=True)
    df_peak["year"] = df_peak["local_time"].dt.year

    yearly = (
        df_peak.groupby("year", as_index=False)
               .apply(lambda g: pd.Series({
                   "avg_peak_rrp": float(np.sum(g["rrp"].to_numpy() * g["demand"].to_numpy()) /
                                         np.sum(g["demand"].to_numpy()))
               }))
               .reset_index(drop=True)
    )

    obs_2024 = float(yearly.loc[yearly["year"] == 2024, "avg_peak_rrp"].iloc[0])
    obs_2025 = float(yearly.loc[yearly["year"] == 2025, "avg_peak_rrp"].iloc[0])
    return obs_2024, obs_2025

if USE_OBSERVED:
    try:
        RAW_CACHE = "/content/nemosis_cache" if IN_COLAB else "./nemosis_cache"
        start_time = "2024/01/01 00:00:00"
        end_time   = "2026/01/01 00:00:00"  # end exclusive -> covers 2024 and 2025

        obs_peak_2024, obs_peak_2025 = compute_observed_peak_prices_local_4_8pm(
            start_time=start_time,
            end_time=end_time,
            raw_cache_dir=RAW_CACHE
        )

        print("Observed NEM demand-weighted peak RRP (LOCAL time 4–8pm):")
        print(f"  2024: {obs_peak_2024:.2f} AUD/MWh")
        print(f"  2025: {obs_peak_2025:.2f} AUD/MWh")

    except Exception as e:
        print("WARNING: Could not compute observed peak prices via NEMOSIS.")
        print("Reason:", repr(e))
        print("Falling back to manual peak targets (edit as needed).")
        USE_OBSERVED = False

if not USE_OBSERVED:
    obs_peak_2024 = 155.0
    obs_peak_2025 = 155.0
    print("Using manual peak targets (LOCAL 4–8pm):")
    print(f"  2024 target: {obs_peak_2024:.2f} AUD/MWh")
    print(f"  2025 target: {obs_peak_2025:.2f} AUD/MWh")

# -----------------------

def g_R_annual(year):
    if year <= 2030:
        return 0.08
    elif year <= 2035:
        return 0.06
    elif year <= 2040:
        return 0.04
    else:
        return 0.03

def g_S_annual(year):
    if year <= 2030:
        return 0.37
    elif year <= 2035:
        return 0.12
    elif year <= 2040:
        return 0.06
    else:
        return 0.03

# -----------------------

num_months = 12
K_R = np.zeros((Y, num_months))
K_S = np.zeros((Y, num_months))

K_R[0, 0] = K_R0
K_S[0, 0] = K_S0

for y_idx in range(Y):
    year = years[y_idx]
    gR_ann = g_R_annual(year)
    gS_ann = g_S_annual(year)

    gR_month = (1.0 + gR_ann)**(1.0 / 12.0) - 1.0
    gS_month = (1.0 + gS_ann)**(1.0 / 12.0) - 1.0

    for m in range(num_months):
        if y_idx == 0 and m == 0:
            continue
        if m == 0:
            prev_y, prev_m = y_idx - 1, num_months - 1
        else:
            prev_y, prev_m = y_idx, m - 1

        K_R[y_idx, m] = K_R[prev_y, prev_m] * (1.0 + gR_month)
        K_S[y_idx, m] = K_S[prev_y, prev_m] * (1.0 + gS_month)

K_R_year_end = K_R[:, -1]
K_S_year_end = K_S[:, -1]

# -----------------------

month_lengths = np.array([31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31])
assert month_lengths.sum() == 365
month_of_day = np.repeat(np.arange(num_months), month_lengths)

# -----------------------

def renewable_surplus(rho, K_R_y_m, D1):
    return np.maximum(rho * K_R_y_m - D1, 0.0)

def storage_policy(s, K_S_y_m):
    x = np.minimum(K_S_y_m, s)
    y = eta * x
    return x, y

def residual_peak(D2, y):
    return np.maximum(D2 - y, 0.0)

def mean_p2_given_lambda(lam, r_values):
    return float(np.mean(np.minimum(c_F + lam * r_values, p_cap)))

# -----------------------

epsD = rng.normal(loc=0.0, scale=0.05, size=(Y, days_per_year))
epsD = np.clip(epsD, -0.9, 0.9)

rho_draws = rng.beta(3.0, 3.0, size=(Y, days_per_year))

# -----------------------

r_mat = np.zeros((Y, days_per_year))
y_mat = np.zeros((Y, days_per_year))

for y_idx in range(Y):
    for d in range(days_per_year):
        m = month_of_day[d]
        KR = K_R[y_idx, m]
        KS = K_S[y_idx, m]

        e = epsD[y_idx, d]
        D2 = D2_2024 * max(0.5, 1.0 + e)
        D1 = D1_2024 * max(0.5, 1.0 + 0.5 * e)

        rho = rho_draws[y_idx, d]

        s = renewable_surplus(rho, KR, D1)
        _, y = storage_policy(s, KS)
        r = residual_peak(D2, y)

        r_mat[y_idx, d] = r
        y_mat[y_idx, d] = y

# -----------------------

target_avg_2425 = 0.5 * (obs_peak_2024 + obs_peak_2025)
r_2425 = np.concatenate([r_mat[0, :], r_mat[1, :]])

def calibrate_lambda_bisection(target, r_values, lam_low=0.0, lam_high=2000.0, iters=80):
    def f(lam):
        return mean_p2_given_lambda(lam, r_values)

    low, high = lam_low, lam_high
    while f(high) < target and high < 1e6:
        high *= 2.0

    if f(high) < target:
        print("WARNING: Could not bracket target peak price; returning very large lambda.")
        return high

    for _ in range(iters):
        mid = 0.5 * (low + high)
        if f(mid) < target:
            low = mid
        else:
            high = mid
    return 0.5 * (low + high)

lam = calibrate_lambda_bisection(target_avg_2425, r_2425)

sim_peak_2024 = mean_p2_given_lambda(lam, r_mat[0, :])
sim_peak_2025 = mean_p2_given_lambda(lam, r_mat[1, :])

print("\nCalibrated lambda:")
print(f"  lambda = {lam:.4f}")
print("Peak price fit (LOCAL 4–8pm target vs simulated p2):")
print(f"  2024 target {obs_peak_2024:.2f} vs sim {sim_peak_2024:.2f} AUD/MWh")
print(f"  2025 target {obs_peak_2025:.2f} vs sim {sim_peak_2025:.2f} AUD/MWh")

# -----------------------

E_p1 = np.zeros(Y)
E_p2 = np.zeros(Y)
E_spread = np.zeros(Y)
E_residual = np.zeros(Y)
E_y = np.zeros(Y)
P_cap = np.zeros(Y)
Phi = np.zeros(Y)


RELIABILITY_THRESHOLDS = [150.0, 200.0]   # AUD/MWh (edit as desired)
P_price_le = {thr: np.zeros(Y) for thr in RELIABILITY_THRESHOLDS}


hist_years = [2024, 2030, 2040, 2050]
hist_idx = [int(np.where(years == yy)[0][0]) for yy in hist_years]
p2_samples_by_year = {}

for y_idx, year in enumerate(years):
    r_vals = r_mat[y_idx, :]
    y_vals = y_mat[y_idx, :]

    p1_vals = np.full(days_per_year, c_F)
    p2_vals = np.minimum(c_F + lam * r_vals, p_cap)

    spread_vals = p2_vals - p1_vals
    profit_vals = (p2_vals - c_F) * r_vals

    E_p1[y_idx] = float(np.mean(p1_vals))
    E_p2[y_idx] = float(np.mean(p2_vals))
    E_spread[y_idx] = float(np.mean(spread_vals))
    E_residual[y_idx] = float(np.mean(r_vals))
    E_y[y_idx] = float(np.mean(y_vals))
    Phi[y_idx] = float(np.mean(profit_vals))

    P_cap[y_idx] = float(np.mean(p2_vals >= p_cap - 1e-6))

    for thr in RELIABILITY_THRESHOLDS:
        P_price_le[thr][y_idx] = float(np.mean(p2_vals <= thr))

    if y_idx in hist_idx:
        p2_samples_by_year[year] = p2_vals.copy()

# -----------------------
# Figure 1: prices and spread (show observed 2024–25 points)
# -----------------------
plt.figure(figsize=(7, 4.5))
plt.plot(years, E_p1, label=r'$\mathbb{E}[p_1]$ (off-peak)')
plt.plot(years, E_p2, label=r'$\mathbb{E}[p_2]$ (peak)')
plt.plot(years, E_spread, label=r'$\mathbb{E}[p_2-p_1]$ (spread)', linestyle='--')

plt.scatter([2024, 2025], [obs_peak_2024, obs_peak_2025],
            marker='o', label='Observed peak (local 4–8pm)', zorder=5)

plt.xlabel('Year')
plt.ylabel('Price (AUD/MWh)')
plt.ylim(0, 250)  # cap is 500
plt.title('Average off-peak and peak prices and spread')
plt.legend()
plt.tight_layout()
plt.savefig('nem_sim_prices.pdf', format='pdf', bbox_inches='tight')
plt.close()

# -----------------------
# Figure 2: residual demand and storage discharge
# -----------------------
plt.figure(figsize=(7, 4.5))
plt.plot(years, E_residual, label=r'$\mathbb{E}[r]$ (residual peak)')
plt.plot(years, E_y, label=r'$\mathbb{E}[y]$ (storage discharge)')
plt.xlabel('Year')
plt.ylabel('Energy (fraction of 2024 peak demand)')
plt.title('Expected residual peak demand and storage discharge')
plt.legend()
plt.tight_layout()
plt.savefig('nem_sim_spread_residual.pdf', format='pdf', bbox_inches='tight')
plt.close()

# -----------------------
# Figure 3: cap frequency and capacity ratios (auto-scale)
# -----------------------
cap_max = float(np.max(P_cap))
cap_ylim_top = max(0.02, 1.2 * cap_max)

fig, ax1 = plt.subplots(figsize=(7, 4.5))
ax1.plot(years, P_cap, marker='o', markersize=3, label='Pr(peak price at cap)')
ax1.set_xlabel('Year')
ax1.set_ylabel('Probability of cap')
ax1.set_ylim(0, cap_ylim_top)

ax2 = ax1.twinx()
ax2.plot(years, K_R_year_end / D2_2024, label=r'$K_R/D_2^{2024}$', linestyle='--')
ax2.plot(years, eta * K_S_year_end / D2_2024, label=r'$\eta K_S / D_2^{2024}$', linestyle=':')
ax2.set_ylabel('Capacity relative to 2024 peak')

capratio_max = float(max((K_R_year_end/D2_2024).max(), (eta*K_S_year_end/D2_2024).max()))
ax2.set_ylim(0, 1.1 * capratio_max)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax2.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.title('Cap frequency and capacity build-out')
fig.tight_layout()
plt.savefig('nem_sim_capfreq_storage.pdf', format='pdf', bbox_inches='tight')
plt.close()

# -----------------------
# Figure 4: fossil operating profit
# -----------------------
plt.figure(figsize=(7, 4.5))
plt.plot(years, Phi, label='Fossil gross operating profit')
plt.xlabel('Year')
plt.ylabel('Profit (AUD per normalised day)')
plt.title('Fossil generators: expected daily gross operating profit')
plt.axhline(0.0, color='gray', linewidth=0.8)
plt.legend()
plt.tight_layout()
plt.savefig('nem_sim_profit.pdf', format='pdf', bbox_inches='tight')
plt.close()

# -----------------------
# Figure 5: PRICE-BASED reliability Pr_y{p2 <= threshold} (mathtext-safe)
# -----------------------
all_rel = np.vstack([P_price_le[thr] for thr in RELIABILITY_THRESHOLDS])
rel_min = float(np.min(all_rel))
rel_max = float(np.max(all_rel))
pad = 0.02
ylo = max(-0.02, rel_min - pad)
yhi = min(1.02, rel_max + pad)

plt.figure(figsize=(7, 4.5))
for thr in RELIABILITY_THRESHOLDS:
    # Use \leq (mathtext supports it; \le can fail depending on backend)
    plt.plot(
        years,
        P_price_le[thr],
        marker='o',
        markersize=3,
        label=rf'$\Pr_y\{{p_2 \leq {thr:.0f}\}}$'
    )
plt.xlabel('Year')
plt.ylabel('Probability')
plt.ylim(ylo, yhi)
plt.title('Price-based reliability: probability peak price is below threshold')
plt.legend()
plt.tight_layout()
plt.savefig('nem_sim_reliability.pdf', format='pdf', bbox_inches='tight')
plt.close()

# -----------------------
# Figure 6: histograms of peak prices for selected years
# -----------------------
fig = plt.figure(figsize=(10, 7.5))
bins = np.linspace(0, p_cap, 35)

for i, yy in enumerate(hist_years, start=1):
    ax = fig.add_subplot(2, 2, i)
    vals = p2_samples_by_year.get(yy, None)
    if vals is None:
        ax.text(0.5, 0.5, f"No data for {yy}", ha='center', va='center')
        ax.set_axis_off()
        continue
    ax.hist(vals, bins=bins, density=True)
    ax.set_title(f"Peak price histogram: {yy}")
    ax.set_xlabel("p2 (AUD/MWh)")
    ax.set_ylabel("Density")
    ax.set_xlim(0, p_cap)

fig.tight_layout()
plt.savefig('nem_sim_histograms.pdf', format='pdf', bbox_inches='tight')
plt.close()

# -----------------------
# Download PDFs in Colab
# -----------------------
output_files = [
    'nem_sim_prices.pdf',
    'nem_sim_spread_residual.pdf',
    'nem_sim_capfreq_storage.pdf',
    'nem_sim_profit.pdf',
    'nem_sim_reliability.pdf',
    'nem_sim_histograms.pdf'
]

print("\nSaved figure files:")
for f in output_files:
    print(" ", f)

if IN_COLAB:
    for f in output_files:
        if os.path.exists(f):
            files.download(f)
